# Трек 1. Агент в классической среде
Исследовательская задача
Выясните, от чего зависит качество, устойчивость или эффективность RL-агента в стандартной среде.

```
Устойчивость: при каком уровне шума или изменении параметров среды качество начинает заметно снижаться?
Обобщение: справляется ли агент с начальными состояниями или условиями, которых не было при обучении?
Поведение и ошибки: одинаковые ли стратегии выучивают агенты с близкой средней наградой и в каких ситуациях они терпят неудачу?
Функция награды: как её изменение влияет на длительность эпизода, расход ресурса или характер действий?
Эффективность обучения: что при фиксированном бюджете взаимодействий со средой выгоднее: один длинный запуск или несколько коротких запусков с выбором модели по отдельным оценочным эпизодам?
Стабильность: насколько итог зависит от seed и почему отдельные запуски оказываются лучше других?
```


«Агент с высокой обобщающей способностью (топовая модель) показывает стабильные результаты, высокую эффективность независимо от seed среды (начальных состояний), в то время как агент с низкой обобщающей способностью (недообученная/переобученная модель) сильно зависим от случайности среды».

In [1]:
import random
import numpy as np
import psutil
import torch
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
%matplotlib inline

In [2]:
if not torch.cuda.is_available():
    # Для маленькой MLP многопоточные накладные расходы на CPU выше выигрыша.
    torch.set_num_threads(1)


def linear_decay(start: float, final: float, step: int, decay_steps: int) -> float:
    fraction = min(max(step / decay_steps, 0.0), 1.0)
    return start + fraction * (final - start)


def is_enough_ram(min_available_gb: float = 0.1) -> bool:
    return psutil.virtual_memory().available >= min_available_gb * 2**30

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# те, у кого есть GPU, но считают несправедливым его использовать, могут раскомментировать:
# device = torch.device('cpu')
device

device(type='cuda')

# Определение агента среды

In [3]:
class DQNAgent(nn.Module):
    def __init__(self, state_shape, n_actions, epsilon=0):
        super().__init__()
        self.epsilon = epsilon
        self.n_actions = n_actions
        self.state_shape = state_shape
        assert len(state_shape) == 1
        state_dim = state_shape[0]

        
        self.fc1=nn.Linear(state_dim,64)
        self.fc2=nn.Linear(64,64)
        self.out=nn.Linear(64,n_actions)


    def forward(self, state_t):
        """Возвращает Q-значения формы [batch_size, n_actions]."""
        
        x=torch.relu(self.fc1(state_t))
        x=torch.relu(self.fc2(x))
        qvalues=self.out(x)
 

        if torch.is_grad_enabled():
            assert qvalues.requires_grad, "qvalues must carry gradients in grad mode"
        assert qvalues.shape == (state_t.shape[0], self.n_actions)
        return qvalues

    @torch.no_grad()
    def get_qvalues(self, states):
        """Вычисляет Q-значения для numpy-массива состояний."""
        model_device = next(self.parameters()).device
        states = torch.as_tensor(np.asarray(states), device=model_device, dtype=torch.float32)
        
        qvalues=self.forward(states)

        return qvalues.cpu().numpy()

    def sample_actions(self, qvalues):
        """Выбирает действия по epsilon-greedy политике."""
        batch_size, n_actions = qvalues.shape
        
        
        best_actions=np.argmax(qvalues,axis=-1)
        random_actions=np.random.choice(n_actions,size=batch_size)

        #Создаём маску масив, которая применяется на весь batch, где тру будем брать лучшее действие, где false случайное действие
        should_explorer=np.random.uniform(0,1,size=batch_size)<=self.epsilon
     
        #Применяем маску
        actions=np.where(should_explorer,random_actions,best_actions)

        return actions



class DuelingDQNHead(nn.Module):
    def __init__(self, input_dim, n_actions):
        super().__init__()
        



        
        self.fc1_dual=nn.Linear(input_dim,128)
        self.fc2_dual=nn.Linear(128,128)
        #self.out=nn.Linear(64,n_actions)
                      
        self.advantage_stream =nn.Linear(128,n_actions)
        self.value_stream = nn.Linear(128,1)


    def forward(self, x):
        
        x=torch.relu(self.fc1_dual(x))
        x=torch.relu(self.fc2_dual(x))
        value=self.value_stream(x)
        advantage=self.advantage_stream(x)
        qvalues=value+(advantage-advantage.mean(dim=1,keepdim=True))

        return qvalues

class DuelingDQN(nn.Module):
    def __init__(self, state_shape, n_actions):
        super().__init__()
        
        state_dim=state_shape[0]
        self.feature_extractor=nn.Sequential(nn.Linear(state_dim,64),nn.ReLU(),)
        self.dueling_head=DuelingDQNHead(64,n_actions)


    def forward(self, x):
        return self.dueling_head(self.feature_extractor(x))

class DuelingDQNAgent(DQNAgent):
    def __init__(self, state_shape, n_actions, epsilon=0):
        nn.Module.__init__(self)
        self.epsilon = epsilon
        self.n_actions = n_actions
        self.state_shape = state_shape
        
        self.network=DuelingDQN(state_shape,n_actions)
    def forward(self, state_t):
       return self.network(state_t)    

### Функция потерь 

In [4]:
def compute_td_loss_double(states, actions, rewards, next_states, is_done,
                           agent, target_network, gamma=0.99,
                           loss_kind="mse", check_shapes=False, device=device):
    states = torch.as_tensor(states, dtype=torch.float32, device=device)
    actions = torch.as_tensor(actions, dtype=torch.int64, device=device)
    rewards = torch.as_tensor(rewards, dtype=torch.float32, device=device)
    next_states = torch.as_tensor(next_states, dtype=torch.float32, device=device)
    is_done = torch.as_tensor(is_done, dtype=torch.bool, device=device)

    predicted_qvalues = agent(states)
    
    # Значения для реальны выполненых действий
    predictions = predicted_qvalues.gather(1, actions.unsqueeze(1)).squeeze(1)   
  

    with torch.no_grad():
            # Сеть Agent выбирает действия 
            next_qvalues_agent = agent(next_states)
            next_act = next_qvalues_agent.argmax(dim=1,keepdim=True)
            #Для оценки ценности выбранного действия используется целевая сеть
            next_qvalues_target=target_network(next_states)
            next_values=next_qvalues_target.gather(1,next_act).squeeze(1)

                 
            targets = (rewards
                       + gamma * next_values * (~is_done).float()
                   )
    if loss_kind=='mse':
        loss=nn.MSELoss()(predictions,targets)
    if loss_kind=='huber':
        loss=nn.HuberLoss()(predictions,targets)
    
    

    if check_shapes:
        assert predictions.shape == rewards.shape
        assert targets.shape == rewards.shape
        assert not targets.requires_grad
    return loss

### Буфер повторов

Интерфейс довольно прост:
* `exp_replay.add(obs, act, rw, next_obs, done)` - сохраняет кортеж (s,a,r,s',done) в буфер
* `exp_replay.sample(batch_size)` - возвращает наблюдения, действия, награды, следующие наблюдения и флаги завершения эпизода для `batch_size` случайных образцов.
* `len(exp_replay)` - возвращает количество элементов, хранящихся в буфере воспроизведения.

In [5]:
class ReplayBuffer(object):
    def __init__(self, size):
        """Создаёт буфер воспроизведения.
        Параметры
        ----------
        size: int
            Максимальное количество переходов, которые можно хранить в буфере.
            При переполнении буфера старые записи удаляются.
        """
        self._storage = []
        self._maxsize = size
        self._next_idx = 0

    def __len__(self):
        return len(self._storage)

    def add(self, obs_t, action, reward, obs_tp1, done):
        data = (obs_t, action, reward, obs_tp1, done)

        if self._next_idx >= len(self._storage):
            self._storage.append(data)
        else:
            self._storage[self._next_idx] = data
        self._next_idx = (self._next_idx + 1) % self._maxsize

    def _encode_sample(self, idxes):
        obses_t, actions, rewards, obses_tp1, dones = [], [], [], [], []
        for i in idxes:
            data = self._storage[i]
            obs_t, action, reward, obs_tp1, done = data
            obses_t.append(np.asarray(obs_t))
            actions.append(np.asarray(action))
            rewards.append(reward)
            obses_tp1.append(np.asarray(obs_tp1))
            dones.append(done)
        return (
            np.array(obses_t),
            np.array(actions),
            np.array(rewards),
            np.array(obses_tp1),
            np.array(dones),
        )

    def sample(self, batch_size):
        """Выбирает случайную партию переходов из буфера.
        Параметры
        ----------
        batch_size: int
            Сколько переходов нужно выбрать.
        Возвращает
        -------
        obs_batch: np.array
            партия наблюдений
        act_batch: np.array
            партия действий, выполненных в состоянии obs_batch
        rew_batch: np.array
            вознаграждения, полученные в результате выполнения act_batch
        next_obs_batch: np.array
            следующие наблюдения после выполнения act_batch
        done_mask: np.array
            done_mask[i] = 1, если выполнение act_batch[i] привело к завершению эпизода,
            и 0 в противном случае.
        """
        idxes = [random.randint(0, len(self._storage) - 1) for _ in range(batch_size)]
        return self._encode_sample(idxes)

### Взаимодействие со средой: `play_and_record`

In [6]:
def play_and_record(initial_state, agent, env, exp_replay, n_steps=1):
    """Делает ровно n_steps взаимодействий и записывает переходы в replay.

    Среда сбрасывается и при terminated, и при truncated. В поле done хранится
    только terminated: ограничение времени не должно обнулять bootstrap-цель.
    """
    state = initial_state
    sum_rewards = 0.0

    
    for n in range(n_steps):
      qvalues = agent.get_qvalues([state])
      action = agent.sample_actions(qvalues)[0]
      #Получаем информацию от среды
      next_state, reward, terminated, truncated,_ = env.step(action)
      sum_rewards += reward
      #Записываем состояние
      exp_replay.add(state,action,reward,next_state,terminated)
      if terminated or truncated:
        state,_=env.reset()
      else:
         state=next_state
   

    return sum_rewards, state

# Определение среды LunarLander